## Importanto Bibliotecas

In [17]:
import os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from datetime import datetime, timedelta

import nilmtk
from nilmtk.utils import print_dict
from nilmtk import DataSet, MeterGroup

## Iniciando a Base de Dados

In [18]:
ukdale = DataSet('../ukdale.h5')

# CASA 1

-   ## Definindo Parâmetros

In [ ]:
#DEFINE O APARELHO A SER UTILIZADO
APARELHO_ATUAL = 'kettle'

#DEFINE OS PARAMETROS PARA A GERACAO DAS IMAGENS
TAXA_AMOSTRAGEM = 6

#DEFINE AS DATAS INICIAIS E FINAIS COMO STRINGS
DATA_INICIAL_CASA1 = '2014-06-29T00:00:00'
DATA_FINAL_CASA1 = '2014-07-10T00:00:00'

# Convertendo as datas
DATA_INICIAL_CASA1 = datetime.strptime(DATA_INICIAL_CASA1, '%Y-%m-%dT%H:%M:%S')
DATA_FINAL_CASA1 = datetime.strptime(DATA_FINAL_CASA1, '%Y-%m-%dT%H:%M:%S')

#DEFINE O INTERVALO DE TEMPO
ukdale.set_window(start=DATA_INICIAL_CASA1, end=DATA_FINAL_CASA1)

#ESCOLHE A CASA 1 E OBTEM OS MEDIDORES
elec = ukdale.buildings[5].elec
mains = elec.mains()

-   ## Definindo Aparelhos

In [ ]:
#DEFINE OS APARELHOS E SEUS RESPECTIVOS LIMIARES EM POTENCIA ATIVA
APARELHOS = ['fridge', 'kettle', 'washing machine', 'dish washer', 'microwave']
LIMIARES = [50, 200, 20, 200, 100]
RATIOS = [0.5, 0.2, 0.5, 0.5, 0.2]
TAMANHO_JANELA = [600, 300, 600, 600, 300] 
AMOSTRAGENS_POR_JANELA = [janela // TAXA_AMOSTRAGEM for janela in TAMANHO_JANELA]
PASSOS = [amostragem // 10 for amostragem in AMOSTRAGENS_POR_JANELA]

indice_aparelho = APARELHOS.index(APARELHO_ATUAL)

grupo_aparelho = (elec.select_using_appliances(type=APARELHOS[indice_aparelho]))
medidor_aparelho = (grupo_aparelho.meters[0])

In [21]:
#OBTEM AS MEDIDAS DE POTENCIA ATIVA DO MEDIDOR CENTRAL
mains_P = next(mains.load(
    physical_quantity='power',
    ac_type='active',
    sample_period=TAXA_AMOSTRAGEM
))

#OBTEM AS MEDIDAS DE POTENCIA APARENTE DO MEDIDOR CENTRAL
mains_S = next(mains.load(
    physical_quantity='power',
    ac_type='apparent',
    sample_period=TAXA_AMOSTRAGEM
))

#OBTEM AS MEDIDAS DE TENSAO DO MEDIDOR CENTRAL
mains_V = next(mains.load(
    physical_quantity='voltage',
    sample_period=TAXA_AMOSTRAGEM
))


df_aparelho = (next(medidor_aparelho.load(
    physical_quantity='power',
    ac_type='active',
    sample_period=TAXA_AMOSTRAGEM
)))


In [22]:
# Remove MultiIndex se existir
if isinstance(df_aparelho.columns, pd.MultiIndex):
    df_aparelho.columns = df_aparelho.columns.get_level_values(-1)

# Constrói o dataframe diretamente
df = pd.concat(
    [mains_P, mains_S, mains_V, df_aparelho],
    axis=1
)

df = df.dropna()

# Opcional (RECOMENDADO): força nomes claros
df.columns = ["P", "S", "V", "P_sub"]


-   ## Janelando dados e Salvando no CSV

In [23]:
import random

def normalizar_int8(lista):
    minimo = min(lista)
    maximo = max(lista)
    delta = maximo - minimo

    lista_int8 = []
    for x in lista:
        # Normaliza para [0,1]
        norm = (x - minimo) / delta
        # Escala para [-128, 127]
        scaled = int(round(norm * 255 - 128))
        # Saturação
        scaled = max(-128, min(127, scaled))
        lista_int8.append(scaled)
    
    return lista_int8

# =============================
# SEGUNDA PASSADA → ARMAZENAR TODAS JANELAS
# =============================

buffer_P, buffer_Q, buffer_I, buffer_sub = [], [], [], []

lista_on = []
lista_off = []
lista_y_on = []
lista_y_off = []

print(f"========== {APARELHO_ATUAL} ==========")

for _, row in df.iterrows():

    P = row.iloc[0]
    S = row.iloc[1]
    V = row.iloc[2]
    P_sub = row.iloc[3]

    I = S / V if V > 0 else 0
    Q = np.sqrt(max(S**2 - P**2, 0))

    buffer_P.append(P)
    buffer_Q.append(Q)
    buffer_I.append(I)
    buffer_sub.append(P_sub)

    if len(buffer_P) == AMOSTRAGENS_POR_JANELA[indice_aparelho]:

        on_ratio = np.mean(np.array(buffer_sub) > LIMIARES[indice_aparelho])
        label = "on" if on_ratio >= RATIOS[indice_aparelho] else "off"

        P_copy = buffer_P.copy()
        Q_copy = buffer_Q.copy()
        I_copy = buffer_I.copy()

        janela = normalizar_int8(P_copy) + normalizar_int8(Q_copy) + normalizar_int8(I_copy)

        if label == "on":
            lista_on.append(janela)
            lista_y_on.append(1)
        else:
            lista_off.append(janela)
            lista_y_off.append(0)

        buffer_P = buffer_P[PASSOS[indice_aparelho]:]
        buffer_Q = buffer_Q[PASSOS[indice_aparelho]:]
        buffer_I = buffer_I[PASSOS[indice_aparelho]:]
        buffer_sub = buffer_sub[PASSOS[indice_aparelho]:]


# =============================
# BALANCEAMENTO ALEATÓRIO
# =============================

random.shuffle(lista_on)
random.shuffle(lista_off)

MAX_POR_CLASSE = min(len(lista_on), len(lista_off))

SPLIT_VALIDATION = 0.6

validation_data = int(MAX_POR_CLASSE * SPLIT_VALIDATION)

lista_on_train = lista_on[:validation_data]
lista_off_train = lista_off[:validation_data]

lista_on_valid = lista_on[validation_data:MAX_POR_CLASSE]
lista_off_valid = lista_off[validation_data:MAX_POR_CLASSE]



lista_y_on_train = lista_y_on[:validation_data]
lista_y_off_train = lista_y_off[:validation_data]

lista_y_on_valid = lista_y_on[validation_data:MAX_POR_CLASSE]
lista_y_off_valid = lista_y_off[validation_data:MAX_POR_CLASSE]

lista_x_train = lista_on_train + lista_off_train
lista_x_valid = lista_on_valid + lista_off_valid

lista_y_train = lista_y_on_train + lista_y_off_train
lista_y_valid = lista_y_on_valid + lista_y_off_valid

print(f"Total treinamento: {len(lista_x_train)}")
print(f"Total validacao: {len(lista_x_valid)}")

========== kettle ==========
Total treinamento: 372
Total validacao: 248


In [ ]:
#SALVA TODOS OS DADOS OBTIDOS EM ARQUIVOS .CSV DE TREINAMENTO E VALIDACAO

shutil.rmtree(f"ModelosTreinados/{APARELHO_ATUAL}", ignore_errors=True)     #APAGA A PASTA CASO JA EXISTA
os.makedirs(f"ModelosTreinados/{APARELHO_ATUAL}", exist_ok=True)
os.makedirs(f"Dados/treinamento/{APARELHO_ATUAL}", exist_ok=True)
os.makedirs(f"Dados/validacao/{APARELHO_ATUAL}", exist_ok=True)

lista_x_train = np.array(lista_x_train)
lista_y_train = np.array(lista_y_train)
lista_x_valid = np.array(lista_x_valid)
lista_y_valid = np.array(lista_y_valid)

np.savetxt(f"Dados/treinamento/{APARELHO_ATUAL}/casa5_x_train.csv", lista_x_train, delimiter=",", fmt="%f")
np.savetxt(f"Dados/treinamento/{APARELHO_ATUAL}/casa5_y_train.csv", lista_y_train, delimiter=",", fmt="%f")
np.savetxt(f"Dados/validacao/{APARELHO_ATUAL}/casa5_x_valid.csv", lista_x_valid, delimiter=",", fmt="%f")
np.savetxt(f"Dados/validacao/{APARELHO_ATUAL}/casa5_y_valid.csv", lista_y_valid, delimiter=",", fmt="%f")